# Combinatorial Optimization with Physics-Inspired Graph Neural Networks

In this notebook we show how to solve combinatorial optimization problems with physics-inspired graph neural networks, as outlined in M. J. A. Schuetz, J. K. Brubaker, H. G. Katzgraber, _Combinatorial Optimization with Physics-Inspired Graph Neural Networks_, [arXiv:2107.01188](https://arxiv.org/abs/2107.01188). 
Here we focus on the canonical maximum independent set (MIS) problem, but our approach can easily be extended to other combinatorial optimization problems. 
For the actual implementation of the graph neural network we use the open-source ```dgl``` library. 

Please note we have provided a `requirements.txt` file, which defines the environment required to run this code. Because some of the packages are not available on default OSX conda channels, we have also provided suggested channels to find them on. These can be distilled into a single line as such:

> conda create -n \<environment_name\> python=3 --file requirements.txt -c conda-forge -c dglteam -c pytorch

In [ ]:
import dgl
import torch
import random
import os
import numpy as np
import networkx as nx
import torch.nn as nn
import torch.nn.functional as F

from collections import OrderedDict, defaultdict
from dgl.nn.pytorch import GraphConv
from itertools import chain, islice, combinations
from networkx.algorithms.approximation.independent_set import maximum_independent_set as mis
from time import time

# MacOS can have issues with MKL. For more details, see
# https://stackoverflow.com/questions/53014306/error-15-initializing-libiomp5-dylib-but-found-libiomp5-dylib-already-initial
os.environ['KMP_DUPLICATE_LIB_OK'] = 'True'

In [4]:
# fix seed to ensure consistent results
seed_value = 1
random.seed(seed_value)        # seed python RNG
np.random.seed(seed_value)     # seed global NumPy RNG
torch.manual_seed(seed_value)  # seed torch RNG

# Set GPU/CPU
TORCH_DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
TORCH_DTYPE = torch.float32
print(f'Will use device: {TORCH_DEVICE}, torch dtype: {TORCH_DTYPE}')

Will use device: cpu, torch dtype: torch.float32


c:\Users\mcasa\anaconda3\envs\rcbi\lib\site-packages\torch\cuda\__init__.py:52: UserWarning: CUDA initialization: Found no NVIDIA driver on your system. Please check that you have an NVIDIA GPU and installed a driver from http://www.nvidia.com/Download/index.aspx (Triggered internally at  ..\c10\cuda\CUDAFunctions.cpp:100.)
  return torch._C._cuda_getDeviceCount() > 0


# Step 0 - Define utility functions

We first load a few general utility functions from ```utils.py``` before defining some helper functions specific to the MIS problem. 

### General utilities

In [5]:
from utils import generate_graph, get_gnn, run_gnn_training, qubo_dict_to_torch, gen_combinations, loss_func

### Problem-specific (MIS) utilities

In [11]:
# helper function to generate Q matrix for Maximum Independent Set problem (MIS)
def gen_q_dict_mis(nx_G, penalty=2):
    """
    Helper function to generate QUBO matrix for MIS as minimization problem.
    
    Input:
        nx_G: graph as networkx graph object (assumed to be unweigthed)
    Output:
        Q_dic: QUBO as defaultdict
    """

    # Initialize our Q matrix
    Q_dic = defaultdict(int)

    # Update Q matrix for every edge in the graph
    # all off-diagonal terms get penalty
    for (u, v) in nx_G.edges:
        Q_dic[(u, v)] = penalty

    # all diagonal terms get -1
    for u in nx_G.nodes:
        Q_dic[(u, u)] = -1

    return Q_dic


# Run classical MIS solver (provided by NetworkX)
def run_mis_solver(nx_graph):
    """
    helper function to run traditional solver for MIS.
    
    Input:
        nx_graph: networkx Graph object
    Output:
        ind_set_bitstring_nx: bitstring solution as list
        ind_set_nx_size: size of independent set (int)
        number_violations: number of violations of ind.set condition
    """
    # compare with traditional solver
    t_start = time()
    ind_set_nx = mis(nx_graph)
    t_solve = time() - t_start
    ind_set_nx_size = len(ind_set_nx)

    # get bitstring list
    nx_bitstring = [1 if (node in ind_set_nx) else 0 for node in sorted(list(nx_graph.nodes))]
    edge_set = set(list(nx_graph.edges))

    # Updated to be able to handle larger scale
    print('Calculating violations...')
    # check for violations
    number_violations = 0
    for ind_set_chunk in gen_combinations(combinations(ind_set_nx, 2), 100000):
        number_violations += len(set(ind_set_chunk).intersection(edge_set))

    return nx_bitstring, ind_set_nx_size, number_violations, t_solve


# Calculate results given bitstring and graph definition, includes check for violations
def postprocess_gnn_mis(best_bitstring, nx_graph):
    """
    helper function to postprocess MIS results

    Input:
        best_bitstring: bitstring as torch tensor
    Output:
        size_mis: Size of MIS (int)
        ind_set: MIS (list of integers)
        number_violations: number of violations of ind.set condition
    """

    # get bitstring as list
    bitstring_list = list(best_bitstring)

    # compute cost
    size_mis = sum(bitstring_list)

    # get independent set
    ind_set = set([node for node, entry in enumerate(bitstring_list) if entry == 1])
    edge_set = set(list(nx_graph.edges))

    print('Calculating violations...')
    # check for violations
    number_violations = 0
    for ind_set_chunk in gen_combinations(combinations(ind_set, 2), 100000):
        number_violations += len(set(ind_set_chunk).intersection(edge_set))

    return size_mis, ind_set, number_violations

# Step 1 - Set hyperparameters

In [12]:
# Graph hypers
n = 100
d = 3
p = None
graph_type = 'reg'

# NN learning hypers #
number_epochs = int(1e5)
learning_rate = 1e-4
PROB_THRESHOLD = 0.5

# Early stopping to allow NN to train to near-completion
tol = 1e-4          # loss must change by more than tol, or trigger
patience = 100    # number early stopping triggers before breaking loop

# Problem size (e.g. graph size)
n = 100

# Establish dim_embedding and hidden_dim values
dim_embedding = int(np.sqrt(n))    # e.g. 10
hidden_dim = int(dim_embedding/2)  # e.g. 5

# Step 2 - Generate random graph

In [22]:
# Constructs a random d-regular or p-probabilistic graph
nx_graph = generate_graph(n=n, d=d, p=p, graph_type=graph_type, random_seed=seed_value)
# get DGL graph from networkx graph, load onto device
graph_dgl = dgl.from_networkx(nx_graph=nx_graph)
graph_dgl = graph_dgl.to(TORCH_DEVICE)

# Construct Q matrix for graph
q_torch = qubo_dict_to_torch(nx_graph, gen_q_dict_mis(nx_graph), torch_dtype=TORCH_DTYPE, torch_device=TORCH_DEVICE)

Generating d-regular graph with n=100, d=3, seed=1


In [23]:
# Visualize graph
pos = nx.kamada_kawai_layout(nx_graph)
nx.draw(nx_graph, pos, with_labels=True, node_color=[[.7, .7, .7]])

TypeError: '_AxesStack' object is not callable

<Figure size 640x480 with 0 Axes>

# Step 3 - Set up optimizer/GNN architecture

In [16]:
# Establish pytorch GNN + optimizer
opt_params = {'lr': learning_rate}
gnn_hypers = {
    'dim_embedding': dim_embedding,
    'hidden_dim': hidden_dim,
    'dropout': 0.0,
    'number_classes': 1,
    'prob_threshold': PROB_THRESHOLD,
    'number_epochs': number_epochs,
    'tolerance': tol,
    'patience': patience
}

net, embed, optimizer = get_gnn(n, gnn_hypers, opt_params, TORCH_DEVICE, TORCH_DTYPE)

# For tracking hyperparameters in results object
gnn_hypers.update(opt_params)

# Step 4 - Run GNN training

In [18]:
print('Running GNN...')
gnn_start = time()

_, epoch, final_bitstring, best_bitstring = run_gnn_training(
    q_torch, graph_dgl, net, embed, optimizer, gnn_hypers['number_epochs'],
    gnn_hypers['tolerance'], gnn_hypers['patience'], gnn_hypers['prob_threshold'])

gnn_time = time() - gnn_start

Running GNN...
Epoch: 0, Loss: 44.144874572753906
Epoch: 1000, Loss: 16.63824462890625
Epoch: 2000, Loss: 4.805368900299072
Epoch: 3000, Loss: 1.108972430229187
Epoch: 4000, Loss: -0.9390431046485901
Epoch: 5000, Loss: -3.6300532817840576
Epoch: 6000, Loss: -11.532605171203613
Epoch: 7000, Loss: -21.94563865661621
Epoch: 8000, Loss: -31.17128562927246
Epoch: 9000, Loss: -36.29430389404297
Epoch: 10000, Loss: -39.04312515258789
Epoch: 11000, Loss: -40.34957504272461
Epoch: 12000, Loss: -40.70170593261719
Epoch: 13000, Loss: -40.84800338745117
Stopping early on epoch 13250 (patience: 100)
GNN training (n=100) took 39.61
GNN final continuous loss: -40.87047576904297
GNN best continuous loss: -40.87047576904297


# Step 5 - Post-process GNN results

In [24]:
final_loss = loss_func(final_bitstring.float(), q_torch)
final_bitstring_str = ','.join([str(x) for x in final_bitstring])

# Process bitstring reported by GNN
size_mis, ind_set, number_violations = postprocess_gnn_mis(best_bitstring, nx_graph)
gnn_tot_time = time() - gnn_start

print(f'Independence number found by GNN is {size_mis} with {number_violations} violations')
print(f'Took {round(gnn_tot_time, 3)}s, model training took {round(gnn_time, 3)}s')

Calculating violations...
Independence number found by GNN is 41 with 0 violations
Took 113.565s, model training took 39.613s


In [25]:
# Visualize result
# Note no light-blue nodes are connected by an edge
color_map = ['orange' if (best_bitstring[node]==0) else 'lightblue' for node in nx_graph.nodes]
nx.draw(nx_graph, pos, with_labels=True, node_color=color_map)

TypeError: '_AxesStack' object is not callable

<Figure size 640x480 with 0 Axes>

# Step 6 - (optional) Compare to classical solver

In [26]:
# run solver
print(f'Running built-in MIS solver (n={n}).')
start = time()
ind_set_bitstring_nx, ind_set_nx_size, nx_number_violations, t_solve = run_mis_solver(nx_graph)
end = time()
runtime_sol = end - start
print(f'Independence number found by nx solver is {ind_set_nx_size} with {nx_number_violations} violations.')
print(f'MIS solver took {round(runtime_sol, 3)}s')

Running built-in MIS solver (n=100).
Calculating violations...
Independence number found by nx solver is 36 with 0 violations.
MIS solver took 0.35s


# Step 7 - Improvements


Generate graphs of different nodes and dimensions, for comparing the performance of the classical solver against the GNN

In [ ]:
from itertools import product
import matplotlib.pyplot as plt
import networkx as nx
import json
from time import time

node_sizes = [10, 50, 100, 250, 500]  # Different graph sizes (number of nodes)
dimensions = [3,4, 5, 6]  # d for d-regular graphs
epochs = 10000
patience = 10
tol = 1e-4
prob_threshold = 0.5

# GNN hyperparameters
gnn_hypers = {
    'dim_embedding': 16,
    'hidden_dim': 32,
    'dropout': 0.5,
    'number_classes': 1,
}
opt_params = {'lr': 0.01}

# Results storage
gnn_results = []
classical_results = []

# Create a figure with enough subplots for the number of graphs
num_graphs = len(node_sizes) * len(dimensions)
fig, axes = plt.subplots(5, 5, figsize=(20, 10))  # Adjust rows/columns if needed
axes = axes.flatten()  # Flatten for easier access

# Ensure we don't exceed available subplots
if num_graphs > len(axes):
    raise ValueError("Too many graphs for the subplot grid. Adjust the layout!")

# Loop through graph configurations
for i, (n, d) in enumerate(product(node_sizes, dimensions)):
    print(f"\n\n-----------Processing graph with {n} nodes and degree {d}---------------")

    # Generate graph
    nx_graph = generate_graph(n=n, d=d, graph_type='reg', random_seed=0)
    graph_dgl = dgl.from_networkx(nx_graph)
    graph_dgl = graph_dgl.to(TORCH_DEVICE)

    # Generate QUBO matrix
    q_torch = qubo_dict_to_torch(nx_graph, gen_q_dict_mis(nx_graph), torch_dtype=TORCH_DTYPE, torch_device=TORCH_DEVICE)

    # Visualize graph
    pos = nx.kamada_kawai_layout(nx_graph)
    ax = axes[i]  # Get the current subplot axis
    nx.draw(nx_graph, pos, with_labels=True, node_color=[[0.7, 0.7, 0.7]], ax=ax)
    ax.set_title(f"Graph with {n} nodes, d={d}")

    # Initialize GNN, embeddings, and optimizer
    net, embed, optimizer = get_gnn(
        n_nodes=n,
        gnn_hypers=gnn_hypers,
        opt_params=opt_params,
        torch_device=TORCH_DEVICE,
        torch_dtype=TORCH_DTYPE,
    )

    # Train GNN
    start = time()
    net, epoch, final_bitstring, best_bitstring = run_gnn_training(
        q_torch, graph_dgl, net, embed, optimizer, epochs, tol, patience, prob_threshold
    )
    end = time()

    # Postprocess results for GNN
    size_mis, ind_set, violations = postprocess_gnn_mis(best_bitstring, nx_graph)

    print(f'Independence number found by GNN is {size_mis} with {violations} violations')

    # Store GNN results
    gnn_result = {
        'nodes': int(n),
        'degree': int(d),
        'size_mis': int(size_mis),
        'violations': int(violations),
        'time': end - start,  # Time taken for GNN
    }
    gnn_results.append(gnn_result)

    # Now run classical solver
    print(f'Running built-in MIS solver (n={n})...')
    start = time()
    ind_set_bitstring_nx, ind_set_nx_size, nx_number_violations, t_solve = run_mis_solver(nx_graph)
    end = time()

    print(f'Independence number found by MIS Solver is {ind_set_nx_size} with {nx_number_violations} violations')

    # Store classical solver results
    classical_result = {
        'nodes': int(n),
        'degree': int(d),
        'size_mis': int(ind_set_nx_size),
        'violations': int(nx_number_violations),
        'time': t_solve,  # Time taken for classical solver
    }
    classical_results.append(classical_result)

# Save results to JSON file
with open('results/gnn_training_results.json', 'w') as gnn_f:
    json.dump(gnn_results, gnn_f, indent=4)

with open('results/classical_solver_results.json', 'w') as classical_f:
    json.dump(classical_results, classical_f, indent=4)

# Adjust layout and show plots
plt.tight_layout()
plt.show()

print("Training completed. Results saved to 'gnn_training_results.json' and 'classical_solver_results.json'.")




-----------Processing graph with 10 nodes and degree 3---------------
Generating d-regular graph with n=10, d=3, seed=0
Epoch: 0, Loss: 9.268771171569824
Epoch: 1000, Loss: -3.9919209480285645
Stopping early on epoch 1491 (patience: 10)
GNN training (n=10) took 4.421
GNN final continuous loss: -3.9999876022338867
GNN best continuous loss: -4.0
Calculating violations...
Independence number found by GNN is 4 with 0 violations
Running built-in MIS solver (n=10)...
Calculating violations...
Independence number found by MIS Solver is 4 with 0 violations


-----------Processing graph with 10 nodes and degree 4---------------
Generating d-regular graph with n=10, d=4, seed=0
Epoch: 0, Loss: 5.376877784729004
Stopping early on epoch 57 (patience: 10)
GNN training (n=10) took 0.17
GNN final continuous loss: 3.512501962177339e-06
GNN best continuous loss: -3.0416553272516467e-05
Calculating violations...
Independence number found by GNN is 0 with 0 violations
Running built-in MIS solver (n=10)

# Step 8 - Uncertainty calculation

Generate a graph with 100 nodes and dimension 3. Perform training to obtaint the MIS 50 times

In [39]:
import json
import numpy as np
from time import time

# Graph hypers
n = 200
d = 3
p = None
graph_type = 'reg'
seed_value = 42  # Set a random seed for reproducibility

# NN learning hypers
number_epochs = 10000
learning_rate = 1e-4
PROB_THRESHOLD = 0.5

# Early stopping to allow NN to train to near-completion
tol = 1e-4  # loss must change by more than tol, or trigger
patience = 100  # number early stopping triggers before breaking loop


# Establish dim_embedding and hidden_dim values
dim_embedding = int(np.sqrt(n))    # e.g., 10
hidden_dim = int(dim_embedding / 2)  # e.g., 5

# Generate the graph
nx_graph = generate_graph(n=n, d=d, p=p, graph_type=graph_type, random_seed=seed_value)

# Convert to DGL and move to device
graph_dgl = dgl.from_networkx(nx_graph=nx_graph)
graph_dgl = graph_dgl.to(TORCH_DEVICE)

# Generate QUBO matrix
q_torch = qubo_dict_to_torch(nx_graph, gen_q_dict_mis(nx_graph), torch_dtype=TORCH_DTYPE, torch_device=TORCH_DEVICE)

# GNN hyperparameters
opt_params = {'lr': learning_rate}
gnn_hypers = {
    'dim_embedding': dim_embedding,
    'hidden_dim': hidden_dim,
    'dropout': 0.0,
    'number_classes': 1,
    'prob_threshold': PROB_THRESHOLD,
    'number_epochs': number_epochs,
    'tolerance': tol,
    'patience': patience
}

# Initialize results list
results = []

# Perform the training multiple times
for i in range(50):
    net, embed, optimizer = get_gnn(n, gnn_hypers, opt_params, TORCH_DEVICE, TORCH_DTYPE)

    # For tracking hyperparameters in results object
    gnn_hypers.update(opt_params)

    print(f"\n\n------------ Running for {i + 1} time ------------")
    gnn_start = time()

    # Training
    _, epoch, final_bitstring, best_bitstring = run_gnn_training(
        q_torch, graph_dgl, net, embed, optimizer,
        gnn_hypers['number_epochs'],
        gnn_hypers['tolerance'],
        gnn_hypers['patience'],
        gnn_hypers['prob_threshold']
    )

    gnn_time = time() - gnn_start

    # Final loss
    final_loss = loss_func(final_bitstring.float(), q_torch)
    final_bitstring_str = ','.join([str(x) for x in final_bitstring])

    # Postprocess results
    size_mis, ind_set, number_violations = postprocess_gnn_mis(best_bitstring, nx_graph)

    print(f'Independence number found by GNN is {size_mis} with {number_violations} violations')
    print(f'Took {round(gnn_tot_time, 3)}s, model training took {round(gnn_time, 3)}s')

    # Store results for this run
    run_result = {
        'run': i + 1,
        'size_mis': int(size_mis),
        'violations': int(number_violations),
        'final_loss': float(final_loss.item()),
        'training_time': round(gnn_time, 4),
    }

    results.append(run_result)

# Save results to JSON
with open('uncertainty.json', 'w') as f:
    json.dump(results, f, indent=4)p

print("\nTraining completed. Results saved to 'uncertainty.json'.")


Generating d-regular graph with n=200, d=3, seed=42


------------ Running for 1 time ------------
Epoch: 0, Loss: 68.48816680908203
Epoch: 1000, Loss: 19.976551055908203
Epoch: 2000, Loss: 4.586960315704346
Epoch: 3000, Loss: 0.6417959928512573
Epoch: 4000, Loss: -1.6255892515182495
Epoch: 5000, Loss: -6.196683406829834
Epoch: 6000, Loss: -20.922996520996094
Epoch: 7000, Loss: -50.16254806518555
Epoch: 8000, Loss: -70.39984893798828
Epoch: 9000, Loss: -81.099609375
GNN training (n=200) took 33.977
GNN final continuous loss: -84.45335388183594
GNN best continuous loss: -84.45335388183594
Calculating violations...
Independence number found by GNN is 86 with 0 violations
Took 61.756s, model training took 33.977s


------------ Running for 2 time ------------
Epoch: 0, Loss: 83.30376434326172
Epoch: 1000, Loss: 32.13877487182617
Epoch: 2000, Loss: 7.4888997077941895
Epoch: 3000, Loss: 1.372482180595398
Epoch: 4000, Loss: -0.5996257662773132
Epoch: 5000, Loss: -1.895744800567627
Epoch: 600

In [40]:
import json
import numpy as np
from scipy import stats

# Load the uncertainty.json file
with open('uncertainty.json', 'r') as file:
    results = json.load(file)

# Extract size_mis values
size_mis_values = [run['size_mis'] for run in results]

# Convert to numpy array for calculations
size_mis_array = np.array(size_mis_values)

# Calculate basic statistics
mean_size_mis = np.mean(size_mis_array)
std_size_mis = np.std(size_mis_array, ddof=1)
variance_size_mis = np.var(size_mis_array, ddof=1)

# 95% Confidence Interval
confidence_level = 0.95
n = len(size_mis_array)
confidence_interval = stats.t.interval(
    confidence_level, 
    df=n-1, 
    loc=mean_size_mis, 
    scale=std_size_mis / np.sqrt(n)
)

# Display the results
print(f"Mean of size_mis: {mean_size_mis:.2f}")
print(f"Standard Deviation of size_mis: {std_size_mis:.2f}")
print(f"Variance of size_mis: {variance_size_mis:.2f}")
print(f"95% Confidence Interval for size_mis: ({confidence_interval[0]:.2f}, {confidence_interval[1]:.2f})")


Mean of size_mis: 82.16
Standard Deviation of size_mis: 3.32
Variance of size_mis: 10.99
95% Confidence Interval for size_mis: (81.22, 83.10)
